# 01. 주차 입차 데이터 전처리 (공연일 vs 비공연일)

이 노트북은 **올팍 공연일 가이드** 웹 데모의 `주차 혼잡` 화면에 들어갈 숫자를 만듭니다.

| 단계 | 하는 일 |
|---|---|
| 0 | 설정 (파일 이름, 컬럼 이름) |
| 1 | CSV 올리기 |
| 2 | 결측치, 중복, 날짜 형식 정리 |
| 3 | 공연 일정과 주차 입차를 **날짜**로 합치기 |
| 4 | 공연일, 비공연일의 **시간대별 평균 입차 대수** 계산 |
| 5 | 결과를 `parking_entry.json` 으로 저장 → GitHub `docs/data/` 에 올리기 |

**사용법**: 위에서부터 셀을 하나씩 `Shift + Enter` 로 실행하세요.
처음에는 `USE_SAMPLE = True` 상태로 끝까지 한번 돌려 보고, 진짜 CSV가 준비되면 `False` 로 바꿉니다.

## 0. 설정
바꿔야 하는 값은 **이 셀에만** 모아 두었습니다. 다른 셀은 고치지 않아도 됩니다.

In [ ]:
# ── 0-1. 샘플 데이터로 연습할지 여부 ─────────────────────────────
# True  : 가짜(합성) 데이터를 자동으로 만들어서 끝까지 실행 (연습용)
# False : 공공데이터포털에서 받은 진짜 CSV를 올려서 실행
USE_SAMPLE = True

# ── 0-2. 파일 이름 (Colab 왼쪽 📁 폴더에 올린 파일 이름과 똑같이) ──
SCHEDULE_FILE = "schedule.csv"   # 공연장 대관(공연) 일정
PARKING_FILE  = "parking.csv"    # 주차장 입출차 기록

# ── 0-3. 컬럼(열) 이름 ─────────────────────────────────────────
# 진짜 CSV를 처음 읽으면 아래 '1단계' 셀에서 컬럼 목록이 출력됩니다.
# 그 이름을 보고 오른쪽 값만 바꾸세요.
SCHEDULE_COLS = {
    "date":  "공연일자",   # 공연 날짜 (예: 2025-08-09, 20250809)
    "venue": "공연장명",   # 공연장 (예: KSPO DOME)
    "title": "공연명",     # 공연 이름 (없으면 None)
}
PARKING_COLS = {
    "datetime": "입차일시",  # 입차 날짜+시각이 한 칸에 있는 경우 (예: 2025-08-09 16:42:10)
    "date":     None,        # 날짜와 시각이 따로 있으면 여기에 날짜 컬럼 이름
    "time":     None,        #   … 그리고 여기에 시각 컬럼 이름 (그때는 datetime 을 None 으로)
    "lot":      "주차장명",  # 주차장 이름 (없으면 None)
}

# ── 0-4. 분석 범위 ─────────────────────────────────────────────
YEAR  = 2025                               # 이 연도 데이터만 사용
HOURS = list(range(10, 23))                # 화면 그래프에 쓸 시간대 (10시~22시)
LOTS  = None                               # 특정 주차장만: ["동2문 주차장", "남2문 주차장"] / 전체: None
VENUES = None                              # 특정 공연장만 공연일로 칠 때: ["KSPO DOME", "올림픽홀"] / 전체: None
NORMAL_WEEKEND_ONLY = True                 # True: 비공연일 중 '주말'만 비교 대상 (공연이 주로 주말이라 더 공정)

OUTPUT_FILE = "parking_entry.json"

## 1. 라이브러리 불러오기, CSV 올리기

In [ ]:
import json, os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colab 에서 실행 중인지 확인 (내 PC 주피터에서도 돌아가도록)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Colab 실행 중:", IN_COLAB, "| pandas", pd.__version__)

In [ ]:
# 샘플 모드: 연습용 가짜 CSV 2개를 만듭니다. (진짜 데이터와 컬럼 이름만 비슷, 값은 무작위)
def make_sample_csvs():
    rng = np.random.default_rng(42)
    days = pd.date_range(f"{YEAR}-06-01", f"{YEAR}-08-31", freq="D")
    # 공연일: 주말 대부분 + 평일 조금
    show_days = [d for d in days if (d.dayofweek >= 5 and rng.random() < 0.55) or rng.random() < 0.08]
    venues = ["KSPO DOME", "올림픽홀", "핸드볼경기장", "88잔디마당"]
    sch = pd.DataFrame({
        "공연일자": [d.strftime("%Y%m%d") for d in show_days],
        "공연장명": rng.choice(venues, len(show_days)),
        "공연명":   [f"샘플 공연 {i+1}" for i in range(len(show_days))],
    })
    # 시간대별 기본 입차량(비공연일)과 공연일 추가량
    base  = {h: v for h, v in zip(range(24), [2,1,1,1,1,2,6,12,20,30,36,44,52,55,58,61,57,52,46,40,31,22,14,6])}
    extra = {h: v for h, v in zip(range(24), [0]*13 + [40,80,150,260,350,310,150,50,20,10,0])}
    lots = ["동2문 주차장", "남2문 주차장", "북2문 주차장"]
    rows = []
    for d in days:
        is_show = d in show_days
        for h in range(24):
            n = rng.poisson(base[h] * (1.2 if d.dayofweek >= 5 else 1.0) + (extra[h] if is_show else 0))
            for _ in range(n):
                t = d + pd.Timedelta(hours=h, minutes=int(rng.integers(60)), seconds=int(rng.integers(60)))
                rows.append((t.strftime("%Y-%m-%d %H:%M:%S"), rng.choice(lots)))
    park = pd.DataFrame(rows, columns=["입차일시", "주차장명"])
    # 일부러 지저분한 행도 섞어 둠 (정리 단계 연습용)
    dirty = pd.DataFrame({"입차일시": [None, "잘못된값", rows[0][0]], "주차장명": ["동2문 주차장", None, rows[0][1]]})
    park = pd.concat([park, dirty], ignore_index=True)
    sch.to_csv(SCHEDULE_FILE, index=False, encoding="utf-8-sig")
    park.to_csv(PARKING_FILE, index=False, encoding="cp949")   # 공공데이터포털 파일은 cp949 인 경우가 많음
    print(f"샘플 생성: {SCHEDULE_FILE} {len(sch)}행, {PARKING_FILE} {len(park):,}행")

if USE_SAMPLE:
    make_sample_csvs()
elif IN_COLAB and not (os.path.exists(SCHEDULE_FILE) and os.path.exists(PARKING_FILE)):
    print("CSV 파일 2개를 선택해 올려 주세요 (파일 이름은 0단계 설정과 같아야 합니다)")
    files.upload()

In [ ]:
# 한글 CSV 는 인코딩이 제각각이라 순서대로 시도합니다.
def read_csv_kr(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc, dtype=str)   # 일단 전부 글자로 읽고, 나중에 형식 변환
            print(f"✔ {path} 읽기 성공 (인코딩 {enc}) → {len(df):,}행, 컬럼: {list(df.columns)}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"{path} 인코딩을 알 수 없습니다")

schedule_raw = read_csv_kr(SCHEDULE_FILE)
parking_raw  = read_csv_kr(PARKING_FILE)
display(schedule_raw.head())
display(parking_raw.head())

## 2. 정리하기 (결측치, 중복, 날짜 형식)
- 컬럼 이름을 영어 표준 이름으로 바꿔 이후 코드가 데이터마다 달라지지 않게 합니다.
- 날짜로 바꿀 수 없는 값, 비어 있는 값은 지웁니다. 몇 행을 지웠는지 꼭 출력해서 기획서에 적을 수 있게 합니다.

In [ ]:
def check_cols(df, mapping, name):
    missing = [v for v in mapping.values() if v and v not in df.columns]
    if missing:
        raise KeyError(f"{name}에 {missing} 컬럼이 없습니다. 0단계 설정의 컬럼 이름을 고쳐 주세요. 실제 컬럼: {list(df.columns)}")

# ── 공연 일정 ──
check_cols(schedule_raw, SCHEDULE_COLS, "공연 일정")
sch = schedule_raw.rename(columns={v: k for k, v in SCHEDULE_COLS.items() if v}).copy()
sch["date"] = pd.to_datetime(sch["date"].str.strip(), errors="coerce", format="mixed").dt.normalize()
n0 = len(sch)
sch = sch.dropna(subset=["date"]).drop_duplicates()
sch = sch[sch["date"].dt.year == YEAR]
if VENUES and "venue" in sch:
    sch = sch[sch["venue"].str.strip().isin(VENUES)]
print(f"공연 일정: {n0}행 → {len(sch)}행 (날짜 오류, 중복, 다른 연도 제거)")

# ── 주차 입차 ──
check_cols(parking_raw, PARKING_COLS, "주차 입차")
park = parking_raw.rename(columns={v: k for k, v in PARKING_COLS.items() if v}).copy()
if PARKING_COLS["datetime"] is None:          # 날짜, 시각이 따로 있는 경우 합치기
    park["datetime"] = park["date"].str.strip() + " " + park["time"].str.strip()
park["datetime"] = pd.to_datetime(park["datetime"], errors="coerce", format="mixed")
n0 = len(park)
park = park.dropna(subset=["datetime"]).drop_duplicates()
park = park[park["datetime"].dt.year == YEAR]
if LOTS and "lot" in park:
    park = park[park["lot"].str.strip().isin(LOTS)]
print(f"주차 입차: {n0:,}행 → {len(park):,}행 (결측, 날짜 오류, 중복, 다른 연도 제거)")

park["date"] = park["datetime"].dt.normalize()
park["hour"] = park["datetime"].dt.hour
print("기간:", park["date"].min().date(), "~", park["date"].max().date())

## 3. 공연 일정과 주차 입차를 날짜로 합치기
하루에 공연이 여러 개면 날짜 하나로 묶은 뒤(공연 수, 공연장 목록) 주차 데이터에 붙입니다.

In [ ]:
show_by_day = (sch.groupby("date")
                  .agg(show_count=("date", "size"),
                       venues=("venue", lambda s: ", ".join(sorted(set(s)))) if "venue" in sch else ("date", "size"))
                  .reset_index())

# 주차 데이터가 있는 날짜 전체 목록에 공연 정보를 왼쪽 병합(left merge)
days = pd.DataFrame({"date": sorted(park["date"].unique())})
days = days.merge(show_by_day, on="date", how="left")
days["show_count"] = days["show_count"].fillna(0).astype(int)
days["is_show"] = days["show_count"] > 0
days["is_weekend"] = days["date"].dt.dayofweek >= 5
print(f"전체 {len(days)}일 중 공연일 {days['is_show'].sum()}일, 비공연일 {(~days['is_show']).sum()}일")
days.head(10)

## 4. 시간대별 평균 입차 대수
날짜×시간 표를 만든 뒤(입차가 0대인 시간도 0으로 채움) 공연일, 비공연일 각각 평균을 냅니다.

In [ ]:
# 날짜 × 시간(0~23) 입차 대수 표
daily_hourly = (park.groupby(["date", "hour"]).size()
                    .unstack(fill_value=0)
                    .reindex(columns=range(24), fill_value=0))

show_days   = days.loc[days["is_show"], "date"]
normal_mask = ~days["is_show"] & (days["is_weekend"] if NORMAL_WEEKEND_ONLY else True)
normal_days = days.loc[normal_mask, "date"]

avg_show   = daily_hourly.loc[show_days].mean().round().astype(int)
avg_normal = daily_hourly.loc[normal_days].mean().round().astype(int)

result = pd.DataFrame({"공연일": avg_show[HOURS], "비공연일": avg_normal[HOURS]})
result["차이(배)"] = (result["공연일"] / result["비공연일"].replace(0, np.nan)).round(1)
print(f"공연일 {len(show_days)}일 평균 vs 비공연일{'(주말)' if NORMAL_WEEKEND_ONLY else ''} {len(normal_days)}일 평균")
peak = result["공연일"].idxmax()
print(f"공연일 최대 혼잡: {peak}시 평균 {result.loc[peak, '공연일']}대 (비공연일 {result.loc[peak, '비공연일']}대)")
result

In [ ]:
# 그래프로 눈으로 확인 (Colab 기본 글꼴은 한글이 깨져서 영어로 표기)
ax = result[["비공연일", "공연일"]].set_axis(["Normal day", "Show day"], axis=1).plot.bar(
    figsize=(9, 3.5), color=["#b9c7cc", "#0f4c5c"], rot=0)
ax.set_xlabel("Hour"); ax.set_ylabel("Avg cars entering"); ax.set_title(f"Olympic Park parking entries by hour ({YEAR})")
plt.tight_layout(); plt.show()

## 5. JSON 으로 저장
웹 데모(`docs/index.html`)가 읽는 형식 그대로 저장합니다.
`hours`, `entry_show`, `entry_norm` 이름은 바꾸지 마세요. 웹 화면이 이 이름으로 찾습니다.

In [ ]:
out = {
    "hours":      HOURS,
    "entry_show": [int(v) for v in result["공연일"]],
    "entry_norm": [int(v) for v in result["비공연일"]],
    "meta": {
        "year": YEAR,
        "show_days": int(len(show_days)),
        "normal_days": int(len(normal_days)),
        "normal_weekend_only": NORMAL_WEEKEND_ONLY,
        "lots": LOTS or "전체",
        "sample_data": USE_SAMPLE,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    },
}
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print(open(OUTPUT_FILE, encoding="utf-8").read())

if IN_COLAB:
    files.download(OUTPUT_FILE)   # 내 PC 로 내려받기

## 5-1. 확인용 CSV 저장 (전처리 전, 후)
팀원이 엑셀로 열어 볼 수 있게 CSV 로도 저장합니다. 한글이 안 깨지도록 **UTF-8 BOM(utf-8-sig)** 으로 씁니다.

| 구분 | 파일 | 내용 |
|---|---|---|
| 전처리 전 (원본) | 내가 올린 `schedule.csv`, `parking.csv` | 용량이 커서 GitHub 에는 안 올림. 공공데이터포털 링크만 `data/05_주차장/README.md` 에 기록 |
| 전처리 후 (결과) | `parking_entry_hourly.csv` | 시간대별 평균 입차 대수 (공연일, 비공연일, 차이) = 화면 그래프 숫자 |
| 전처리 후 (결과) | `parking_daily.csv` | 날짜별 입차 합계, 공연 수, 공연장, 공연일 여부 |
| 정리 기록 | `parking_cleaning_log.csv` | 단계별 행 수 (정리 전 → 정리 후). 기획서 전처리 설명에 그대로 쓰기 |

샘플 모드(`USE_SAMPLE = True`)로 돌리면 파일 이름 끝에 `_SAMPLE` 이 붙습니다. **진짜 데이터가 아니라는 표시**이니 기획서에 쓰지 마세요.
받은 파일은 GitHub `data/05_주차장/processed/` 에 올려 팀과 나눕니다.


In [ ]:
import zipfile
tag = "_SAMPLE" if USE_SAMPLE else ""
note = "합성 샘플 (진짜 데이터 아님)" if USE_SAMPLE else "실제 데이터"

hourly = result.reset_index().rename(columns={"index": "시간"})
hourly.columns = ["시간", "공연일_평균입차", "비공연일_평균입차", "차이_배"]
hourly["데이터구분"] = note

daily_total = park.groupby("date").size().rename("입차합계")
daily = days.merge(daily_total, left_on="date", right_index=True, how="left")
daily = pd.DataFrame({
    "날짜": daily["date"].dt.strftime("%Y-%m-%d"),
    "요일": daily["date"].dt.dayofweek.map(dict(enumerate("월화수목금토일"))),
    "공연일": daily["is_show"].map({True: "공연일", False: "비공연일"}),
    "공연수": daily["show_count"],
    "공연장": daily["venues"].fillna("") if "venues" in daily else "",
    "입차합계": daily["입차합계"].fillna(0).astype(int),
    "데이터구분": note,
})

log = pd.DataFrame([
    {"단계": "공연 일정 원본", "행수": len(schedule_raw)},
    {"단계": "공연 일정 정리 후", "행수": len(sch)},
    {"단계": "주차 입차 원본", "행수": len(parking_raw)},
    {"단계": "주차 입차 정리 후", "행수": len(park)},
    {"단계": "분석 날짜 수", "행수": len(days)},
    {"단계": "공연일 수", "행수": int(len(show_days))},
    {"단계": "비교 비공연일 수", "행수": int(len(normal_days))},
])
log["데이터구분"] = note

saved = []
for name, df in [("parking_entry_hourly", hourly), ("parking_daily", daily), ("parking_cleaning_log", log)]:
    fn = f"{name}{tag}.csv"
    df.to_csv(fn, index=False, encoding="utf-8-sig")
    saved.append(fn)
    print(f"✔ {fn} ({len(df)}행)")
display(hourly)

if IN_COLAB:
    with zipfile.ZipFile(f"parking_csv{tag}.zip", "w") as z:
        for fn in saved:
            z.write(fn)
    files.download(f"parking_csv{tag}.zip")


## 6. GitHub 에 올리기
1. 저장소 페이지 → `docs/data` 폴더로 이동
2. **Add file → Upload files** → 방금 받은 `parking_entry.json` 끌어다 놓기
3. 아래 Commit 메시지에 `주차 입차 JSON 갱신` 처럼 적고 **Commit changes**
4. 1~2분 뒤 GitHub Pages 링크를 새로고침하면 `주차 혼잡` 화면 그래프가 바뀝니다.

> CSV 3개(5-1단계)는 같은 방법으로 `data/05_주차장/processed` 폴더에 올리면 팀원이 엑셀로 볼 수 있습니다.

> 진짜 데이터로 바꿨다면 `meta.sample_data` 가 `false` 인지 꼭 확인하세요.